# NegMerge Tutorial

## 1. Import Requirements

In [1]:
import torch
import os
import json
import argparse
import sys
import timm.data.transforms
import abc
import sys
sys.path.insert(0, '/home/joao/Code/PhD_Modules/DiffProg_and_DL/COMP6258-negmerge/CLIP_MU')

import os
os.environ["HF_CARS_ROOT"] = "/home/joao/Code/PhD_Modules/DiffProg_and_DL/COMP6258-negmerge/datasets_local/stanford_cars_hf"

# optional sanity check
import glob
print(glob.glob(os.path.join(os.environ["HF_CARS_ROOT"], "data", "train-*.parquet"))[:2])


['/home/joao/Code/PhD_Modules/DiffProg_and_DL/COMP6258-negmerge/datasets_local/stanford_cars_hf/data/train-00001-of-00002.parquet', '/home/joao/Code/PhD_Modules/DiffProg_and_DL/COMP6258-negmerge/datasets_local/stanford_cars_hf/data/train-00000-of-00002.parquet']


/home/joao/Code/PhD_Modules/DiffProg_and_DL/COMP6258-negmerge/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if 'ipykernel' in sys.modules:
    sys.argv = ['']

class MaybeToTensor:
    def __call__(self, x):
        return x
timm.data.transforms.MaybeToTensor = MaybeToTensor
device = torch.device("cpu")

## 2. Define Configuration

In [3]:
def parse_arguments():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_location", type=str, default=os.path.expanduser("~/data"), help="The root directory for the datasets.")
    parser.add_argument("--eval-datasets", default=None, type=lambda x: x.split(","), help="Which datasets to use for evaluation. Split by comma, e.g. MNIST,EuroSAT.")
    parser.add_argument("--results_db", type=str, default=None, help="Where to store the results, else does not store")
    parser.add_argument("--model", type=str, default="ViT-B-32", help="The type of model (e.g. RN50, ViT-B-32).")
    parser.add_argument("--save", type=str, default=None, help="Optionally save a _classifier_, e.g. a zero shot classifier or probe.")
    parser.add_argument("--load", type=lambda x: x.split(","), default=None, help="Optionally load a _classifier_, e.g. a zero shot classifier or probe.")
    parser.add_argument("--seed", type=int, default=None, help="Random seed.")
    parser.add_argument("--finetuning_mode", choices=["standard", "linear", "none"], help="Whether to use linearized models or not.")
    parser.add_argument("--n-eval-points", type=int, default=21, help="Number of evaluation points used to find optimal coefficient in task arithmetic.")

    parsed_args = parser.parse_args()
    parsed_args.device = "cuda" if torch.cuda.is_available() else "cpu"

    if parsed_args.load is not None and len(parsed_args.load) == 1:
        parsed_args.load = parsed_args.load[0]
        
    return parsed_args


In [4]:
args = parse_arguments()

args.data_location = "/home/joao/Code/PhD_Modules/DiffProg_and_DL/COMP6258-negmerge/datasets_local"
args.finetuning_mode = "standard"       # "linear" or "standard"
args.model = "ViT-B-32"                 # Backbone
args.results_db = "checkpoints"
args.save = os.path.join(args.results_db, args.finetuning_mode, args.model)
args.openclip_cachedir = os.path.expanduser("~/.cache/open_clip")

dataset = "Cars"                        # Forget set
control_dataset = "ImageNet"            # Retain set

FINETUNED_DIR = "/home/joao/Code/PhD_Modules/DiffProg_and_DL/COMP6258-negmerge/finetuned_them"

with open(os.path.join(FINETUNED_DIR, "zeroshot_accuracies.json")) as f:
    pretrained_accuracies = json.load(f)
negation_accuracies = {}

# Compatibility defaults expected by src/modeling.py and training/eval code
if not hasattr(args, "auto_aug"):
    args.auto_aug = None
if not hasattr(args, "train_dataset"):
    args.train_dataset = None
if not hasattr(args, "batch_size"):
    args.batch_size = 128
if not hasattr(args, "num_workers"):
    args.num_workers = 8
if not hasattr(args, "cache_dir"):
    args.cache_dir = os.path.expanduser("~/.cache")


## 3. Dowload Pretrained and Fine-tuned Weights
- Download Link: https://drive.google.com/drive/u/1/folders/1m1iHi5KoTN1Fg5JqIZxtVP1ZTxgILZyi

In [5]:
pretrained_path = os.path.join(FINETUNED_DIR, 'zeroshot.pt')
finetuned_paths = [
    os.path.join(FINETUNED_DIR, f'clip-vit-b-32_cars_rand-m{m}-n{n}_finetuned.pt')
    for m in range(1, 11)
    for n in range(1, 4)
]


## 4. Define Task Vector Class

In [6]:
class _TaskVector(abc.ABC):
    def __init__(
        self, pretrained_checkpoint=None, finetuned_checkpoint=None, vector=None
    ):
        if vector is not None:
            self.vector = vector
        else:
            assert (
                pretrained_checkpoint is not None and finetuned_checkpoint is not None
            )
            with torch.no_grad():
                if isinstance(pretrained_checkpoint, dict):
                    pretrained_state_dict = pretrained_checkpoint
                else:
                    pretrained_state_dict = self._load_checkpoint(
                        pretrained_checkpoint
                    ).state_dict()

                if isinstance(finetuned_checkpoint, dict):
                    finetuned_state_dict = finetuned_checkpoint
                else:
                    finetuned_state_dict = self._load_checkpoint(
                        finetuned_checkpoint
                    ).state_dict()

                self.vector = {}
                for key in pretrained_state_dict:
                    if pretrained_state_dict[key].dtype == torch.int64:
                        continue
                    if pretrained_state_dict[key].dtype == torch.uint8:
                        continue
                    self.vector[key] = (
                        finetuned_state_dict[key] - pretrained_state_dict[key]
                    )

    @abc.abstractmethod
    def _load_checkpoint(self, checkpoint):
        """Load a checkpoint into a model."""
        raise NotImplementedError

    @abc.abstractmethod
    def _cast_to_same_type(self, other):
        raise NotImplementedError

    def __add__(self, other):
        """Add two task vectors together."""
        other = self._cast_to_same_type(other)
        with torch.no_grad():
            new_vector = {}
            for key in self.vector:
                if key not in other.vector:
                    print(f"Warning, key {key} is not present in both task vectors.")
                    continue
                new_vector[key] = self.vector[key] + other.vector[key]
        return self.__class__(vector=new_vector)

    def __sub__(self, other):
        """Subtract two task vectors."""
        return self.__add__(-other)

    def __radd__(self, other):
        if other is None or isinstance(other, int):
            return self
        return self.__add__(other)

    def __neg__(self):
        """Negate a task vector."""
        with torch.no_grad():
            new_vector = {}
            for key in self.vector:
                new_vector[key] = -self.vector[key]
        return self.__class__(vector=new_vector)

    def __pow__(self, power):
        """Power of a task vector."""
        with torch.no_grad():
            new_vector = {}
            for key in self.vector:
                new_vector[key] = self.vector[key] ** power
        return self.__class__(vector=new_vector)

    def __mul__(self, other):
        """Multiply a task vector by a scalar."""
        with torch.no_grad():
            new_vector = {}
            for key in self.vector:
                new_vector[key] = other * self.vector[key]
        return self.__class__(vector=new_vector)

    def dot(self, other):
        """Dot product of two task vectors."""
        other = self._cast_to_same_type(other)
        with torch.no_grad():
            dot_product = 0.0
            for key in self.vector:
                if key not in other.vector:
                    print(f"Warning, key {key} is not present in both task vectors.")
                    continue
                dot_product += torch.sum(self.vector[key] * other.vector[key])
        return dot_product

    def norm(self):
        """Norm of a task vector."""
        return torch.sqrt(self.dot(self))

    def apply_to(self, pretrained_checkpoint, scaling_coef=1.0):
        """Apply a task vector to a pretrained model."""
        with torch.no_grad():
            pretrained_model = self._load_checkpoint(pretrained_checkpoint)
            new_state_dict = {}
            pretrained_state_dict = pretrained_model.state_dict()
            for key in pretrained_state_dict:
                if key not in self.vector:
                    print(
                        f"Warning: key {key} is present in the pretrained state dict but not in the task vector"  # noqa: E501
                    )
                    continue
                new_state_dict[key] = (
                    pretrained_state_dict[key] + scaling_coef * self.vector[key]
                )
        pretrained_model.load_state_dict(new_state_dict)
        return pretrained_model


class NonLinearTaskVector(_TaskVector):
    """A task vector for nonlinear models."""

    def _load_checkpoint(self, checkpoint):
        """Load a checkpoint into a model."""
        return torch.load(checkpoint, map_location="cpu", weights_only=False)

    def apply_to_nonlinear(self, pretrained_nonlinear_checkpoint, scaling_coef=1.0):
        """Apply a task vector to a nonlinear pretrained model."""
        return self.apply_to(pretrained_nonlinear_checkpoint, scaling_coef)
    
    def _cast_to_same_type(self, other):
        return linear_to_nonlinear(other, self.vector.keys())

def linear_to_nonlinear(linear_task_vector, param_names):
    """Convert a linear task vector to a nonlinear task vector."""
    if isinstance(linear_task_vector, NonLinearTaskVector):
        return linear_task_vector
    else:
        return NonLinearTaskVector(
            vector=linear_task_vector.get_named_parameters(param_names)
        )


## 5. Merge Task Vectors

In [7]:
for idx, finetuned_path in enumerate(finetuned_paths):
    state_dict = torch.load(finetuned_path, map_location=device, weights_only=False)
    state_dict = {k: v.to(device) for k, v in state_dict.items()}
        
    task_vector = (NonLinearTaskVector(pretrained_path, state_dict))

    if idx == 0:
        merged_vector = {k: torch.zeros_like(v) for k, v in task_vector.vector.items()}
        mask = {k: torch.zeros_like(v) for k, v in task_vector.vector.items()}

    for key in task_vector.vector.keys():
        merged_vector[key] += task_vector.vector[key]
        mask[key] += torch.sign(task_vector.vector[key])

for key in torch.load(finetuned_path, map_location=device, weights_only=False).keys():
    consistency_mask = torch.abs(mask[key]) == len(finetuned_paths)
    task_vector.vector[key] = torch.where(consistency_mask, merged_vector[key] / len(finetuned_paths), torch.zeros_like(merged_vector[key]))


## 6. Evaluate

### 6.1. Find Optimal Coefficient

In [8]:
import os
from datasets import load_dataset
from tqdm import tqdm
from huggingface_hub import get_token

OUT_ROOT = "/home/joao/Code/PhD_Modules/DiffProg_and_DL/COMP6258-negmerge/datasets_local/imagenet"
os.makedirs(os.path.join(OUT_ROOT, "val"), exist_ok=True)

token = get_token()
if not token:
    raise RuntimeError(
        "No Hugging Face token found. Run `hf auth login` in your terminal first."
    )

# Validation only: streaming avoids downloading/validating train/test splits
try:
    val_ds = load_dataset(
        "ILSVRC/imagenet-1k",
        split="validation",
        token=token,
        streaming=True,
    )
except Exception as e:
    msg = str(e)
    if "403 Forbidden" in msg or "public gated repositories" in msg:
        raise RuntimeError(
            "HF token lacks gated-dataset permission. Enable 'Access to public gated repositories' "
            "for your token at https://huggingface.co/settings/tokens, then run `hf auth login` again."
        ) from e
    raise

def export_val(ds):
    split_dir = os.path.join(OUT_ROOT, "val")
    os.makedirs(split_dir, exist_ok=True)

    for i, ex in enumerate(tqdm(ds, desc="Export val")):
        img = ex["image"]
        label = int(ex["label"])
        cls_dir = os.path.join(split_dir, f"{label:04d}")
        os.makedirs(cls_dir, exist_ok=True)
        img.save(os.path.join(cls_dir, f"{i:08d}.JPEG"), format="JPEG", quality=95)

export_val(val_ds)
print("Validation export done.")

Export val: 50000it [02:33, 325.16it/s]

Validation export done.


In [9]:
from src.eval import evaluate_task_vector, evaluate_task_vector_at_coef
from src.utils import find_optimal_coef

args.eval_datasets = [dataset + "Val"]
args.control_dataset = control_dataset + "Val"
val_metrics = evaluate_task_vector(
    -task_vector,
    pretrained_path,
    args,
)

optimal_coef = find_optimal_coef(
    val_metrics,
    metric=f"{dataset}Val:top1",
    minimize=True,
    control_metric=f"{control_dataset}Val:top1",
    control_metric_threshold=0.95 * pretrained_accuracies[control_dataset + "Val"],
)

# If no coefficient satisfies the control threshold, fall back to best forget metric.
if optimal_coef is None:
    optimal_coef = min(
        val_metrics.keys(),
        key=lambda c: val_metrics[c][f"{dataset}Val:top1"],
    )
    print(
        "No coefficient satisfied control threshold; "
        f"falling back to lowest {dataset}Val:top1 at coef={optimal_coef:.3f}"
    )

print(f"Selected optimal coefficient: {optimal_coef:.3f}")

Evaluating for scaling coefficient 0.00
Evaluating on CarsVal
Classification head for ViT-B-32 on CarsVal exists at checkpoints/standard/ViT-B-32/head_CarsVal.pt
Loading classification head from checkpoints/standard/ViT-B-32/head_CarsVal.pt


/home/joao/Code/PhD_Modules/DiffProg_and_DL/COMP6258-negmerge/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 8, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
  0%|          | 0/7 [00:00<?, ?it/s]/home/joao/Code/PhD_Modules/DiffProg_and_DL/COMP6258-negmerge/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 8, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, low

Done evaluating on CarsVal. Accuracy: 0.61%
CarsVal Top-1 accuracy: 0.0061
Evaluating on ImageNetVal
Did not find classification head for ViT-B-32 on ImageNetVal at checkpoints/standard/ViT-B-32/head_ImageNetVal.pt, building one from scratch.
Loading ViT-B-32 pre-trained weights.


/home/joao/Code/PhD_Modules/DiffProg_and_DL/COMP6258-negmerge/.venv/lib/python3.12/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Building classification head.


...

Done evaluating on ImageNetVal. Accuracy: 58.15%
ImageNetVal Top-1 accuracy: 0.5815
Evaluating for scaling coefficient 1.00
Evaluating on CarsVal
Classification head for ViT-B-32 on CarsVal exists at checkpoints/standard/ViT-B-32/head_CarsVal.pt
Loading classification head from checkpoints/standard/ViT-B-32/head_CarsVal.pt


100%|██████████| 7/7 [00:02<00:00,  3.00it/s]


Done evaluating on CarsVal. Accuracy: 0.25%
CarsVal Top-1 accuracy: 0.0025
Evaluating on ImageNetVal
Classification head for ViT-B-32 on ImageNetVal exists at checkpoints/standard/ViT-B-32/head_ImageNetVal.pt
Loading classification head from checkpoints/standard/ViT-B-32/head_ImageNetVal.pt


100%|██████████| 391/391 [00:43<00:00,  8.97it/s]

Done evaluating on ImageNetVal. Accuracy: 57.77%
ImageNetVal Top-1 accuracy: 0.5777
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.6332699999999999 threshold
Control metric fell below 0.63326999

### 6.2. Evaluate on the test set with the optimal coefficient.

In [16]:
args.eval_datasets = [dataset]
args.control_dataset = control_dataset

control_eval_dataset = control_dataset
imagenet_train_dir = os.path.join(args.data_location, "imagenet", "train")
train_has_class_dirs = os.path.isdir(imagenet_train_dir) and any(
    os.path.isdir(os.path.join(imagenet_train_dir, d)) for d in os.listdir(imagenet_train_dir)
)
if control_dataset == "ImageNet" and not train_has_class_dirs:
    control_eval_dataset = "ImageNetVal"
    print(
        "ImageNet train split is missing/empty; using ImageNetVal as control dataset for test-time evaluation."
    )

args.control_dataset = control_eval_dataset

if optimal_coef is None:
    if "val_metrics" not in globals() or val_metrics is None:
        raise RuntimeError(
            "optimal_coef is None and val_metrics is unavailable. Re-run Cell 17 first."
        )
    optimal_coef = min(
        val_metrics.keys(),
        key=lambda c: val_metrics[c][f"{dataset}Val:top1"],
    )
    print(
        "No valid control-constrained coefficient in memory; "
        f"using fallback coef={optimal_coef:.3f} from validation metrics."
    )

test_metrics = evaluate_task_vector_at_coef(
    -task_vector,
    pretrained_path,
    args,
    optimal_coef,
)

print("=" * 100)
print(f"Test accuracy: {test_metrics[f'{dataset}:top1']}")

negation_accuracies[dataset] = {
    "test": test_metrics[f"{dataset}:top1"],
    "test_control": test_metrics[f"{control_eval_dataset}:top1"],
    "val": val_metrics,
}

print(negation_accuracies[dataset])

ImageNet train split is missing/empty; using ImageNetVal as control dataset for test-time evaluation.
Evaluating on Cars
Classification head for ViT-B-32 on CarsVal exists at checkpoints/standard/ViT-B-32/head_CarsVal.pt
Loading classification head from checkpoints/standard/ViT-B-32/head_CarsVal.pt


100%|██████████| 63/63 [00:09<00:00,  6.58it/s]


Done evaluating on Cars. Accuracy: 0.58%
Cars Top-1 accuracy: 0.0058
Evaluating on ImageNetVal
Classification head for ViT-B-32 on ImageNetVal exists at checkpoints/standard/ViT-B-32/head_ImageNetVal.pt
Loading classification head from checkpoints/standard/ViT-B-32/head_ImageNetVal.pt


100%|██████████| 391/391 [00:43<00:00,  8.94it/s]

Done evaluating on ImageNetVal. Accuracy: 58.15%
ImageNetVal Top-1 accuracy: 0.5815
Test accuracy: 0.005845044148737719
{'test': 0.005845044148737719, 'test_control': 0.58154, 'val': {np.float64(0.0): {'CarsVal:top1': 0.006142506142506142, 'ImageNetVal:top1': 0.61546}, np.float64(0.05): {'CarsVal:top1': 0.006142506142506142, 'ImageNetVal:top1': 0.6154}, np.float64(0.1): {'CarsVal:top1': 0.006142506142506142, 'ImageNetVal:top1': 0.6146}, np.float64(0.15000000000000002): {'CarsVal:top1': 0.006142506142506142, 'ImageNetVal:top1': 0.61422}, np.float64(0.2): {'CarsVal:top1': 0.006142506142506142, 'ImageNetVal:top1': 0.6135}, np.float64(0.25): {'CarsVal:top1': 0.006142506142506142, 'ImageNetVal:top1': 0.61228}, np.float64(0.30000000000000004): {'CarsVal:top1': 0.006142506142506142, 'ImageNetVal:top1': 0.61126}, np.float64(0.35000000000000003): {'CarsVal:top1': 0.006142506142506142, 'ImageNetVal:top1': 0.60982}, np.float64(0.4): {'CarsVal:top1': 0.007371007371007371, 'ImageNetVal:top1': 0.608